In [1]:
import pandas as pd
import numpy as np 

filtered_dataset = pd.read_csv(
    "data/filtered_dataset.csv"
)

In [2]:
filtered_dataset

,country,year,iso_code,population,gdp,cement_co2,cement_co2_per_capita,co2,co2_growth_abs,co2_growth_prct,...,share_global_other_co2,share_of_temperature_change_from_ghg,temperature_change_from_ch4,temperature_change_from_co2,temperature_change_from_ghg,temperature_change_from_n2o,total_ghg,total_ghg_excluding_lucf,trade_co2,trade_co2_share
0,Australia,1990,AUS,17126304.0,4.641366e+11,3.463,0.202,278.061,21.223,8.263,...,0.666,1.693,0.007,0.008,0.017,0.002,538.479,336.927,-33.380,-12.005
1,Australia,1991,AUS,17353191.0,4.613021e+11,3.183,0.183,279.437,1.376,0.495,...,0.643,1.687,0.007,0.008,0.017,0.002,522.525,339.829,-34.011,-12.171
2,Australia,1992,AUS,17549286.0,4.786337e+11,2.923,0.167,284.433,4.996,1.788,...,0.699,1.680,0.007,0.008,0.017,0.002,568.962,346.607,-36.367,-12.786
3,Australia,1993,AUS,17722905.0,5.015248e+11,3.005,0.170,288.780,4.347,1.528,...,0.810,1.677,0.007,0.008,0.017,0.002,580.666,350.855,-40.004,-13.853
4,Australia,1994,AUS,17897433.0,5.279932e+11,3.484,0.195,293.613,4.833,1.674,...,0.924,1.671,0.007,0.008,0.018,0.002,554.084,353.411,-35.667,-12.148
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
345,United States,2020,USA,339436156.0,1.802736e+13,40.688,0.120,4689.954,-545.958,-10.427,...,30.237,18.090,0.037,0.238,0.286,0.011,5758.456,5309.856,413.131,8.809
346,United States,2021,USA,340161438.0,1.909938e+13,41.312,0.121,5020.111,330.157,7.040,...,31.844,17.980,0.037,0.240,0.288,0.011,6131.129,5675.305,469.552,9.353
347,United States,2022,USA,341534041.0,1.949317e+13,41.884,0.123,5055.403,35.292,0.703,...,30.882,17.870,0.038,0.242,0.291,0.011,6201.504,5728.265,510.586,10.100
348,United States,2023,USA,343477330.0,NaN,40.636,0.118,4918.407,-136.996,-2.710,...,31.847,17.759,0.038,0.245,0.294,0.011,6078.455,5606.243,513.252,10.435


In [3]:
filtered_dataset["decade"] = (
    filtered_dataset["year"] // 10
) * 10

vectorization is done on the entire column year and new column is formed that is decade

In [4]:
filtered_dataset[["year", "decade"]].head(5)

,year,decade
0,1990,1990
1,1991,1990
2,1992,1990
3,1993,1990
4,1994,1990


In [5]:
filtered_dataset["years_since_1990"] = (
    filtered_dataset["year"] - 1990
)

A new feature called `years_since_1990` is created by subtracting 1990 from the `year` column. This converts calendar years into a simple numerical time index, making it easier for regression models to learn temporal relationships while preserving the order of observations.

In [6]:
filtered_dataset[
    ["year","years_since_1990"]
].head(5)

,year,years_since_1990
0,1990,0
1,1991,1
2,1992,2
3,1993,3
4,1994,4


In [7]:
filtered_dataset["co2_5yr_rolling_mean"] = (
    filtered_dataset
    .groupby("country")["co2"]
    .transform(
        lambda x: x.rolling(window=5).mean()
    )
)

##### Creating the 5-Year Rolling Mean Feature

A new feature, co2_5yr_rolling_mean, captures the long-term trend of CO₂ emissions by calculating the average emissions over the current year and the previous four years. The dataset is grouped by country so the rolling mean is computed independently for each country. Using rolling(window=5).mean() smooths short-term fluctuations and highlights long-term trends, making the data more suitable for time-series analysis and machine learning.

In [8]:
filtered_dataset[
    [
        "country",
        "year",
        "co2",
        "co2_5yr_rolling_mean"
    ]
].head()

,country,year,co2,co2_5yr_rolling_mean
0,Australia,1990,278.061,NaN
1,Australia,1991,279.437,NaN
2,Australia,1992,284.433,NaN
3,Australia,1993,288.780,NaN
4,Australia,1994,293.613,284.8648


In [9]:
filtered_dataset["co2_lag1"] = (
    filtered_dataset
    .groupby("country")["co2"]
    .shift(1)
)

filtered_dataset["co2_lag2"] = (
    filtered_dataset
    .groupby("country")["co2"]
    .shift(2)
)

filtered_dataset["co2_lag3"] = (
    filtered_dataset
    .groupby("country")["co2"]
    .shift(3)
)

##### Creating Lag Features

Lag features capture previous years' CO₂ emissions, helping the model learn temporal patterns.

Three lag features are created:

1) co2_lag1: Previous year's CO₂ emission
2) co2_lag2: CO₂ emission from two years ago
3) co2_lag3: CO₂ emission from three years ago

The data is grouped by country, and shift() is used to create these features without mixing countries.

In [10]:
filtered_dataset[
    [
        "country",
        "year",
        "co2",
        "co2_lag1",
        "co2_lag2",
        "co2_lag3"
    ]
].head(15)

,country,year,co2,co2_lag1,co2_lag2,co2_lag3
0,Australia,1990,278.061,NaN,NaN,NaN
1,Australia,1991,279.437,278.061,NaN,NaN
2,Australia,1992,284.433,279.437,278.061,NaN
3,Australia,1993,288.780,284.433,279.437,278.061
4,Australia,1994,293.613,288.780,284.433,279.437
5,Australia,1995,304.962,293.613,288.780,284.433
6,Australia,1996,311.851,304.962,293.613,288.780
7,Australia,1997,320.243,311.851,304.962,293.613
8,Australia,1998,334.038,320.243,311.851,304.962
9,Australia,1999,343.876,334.038,320.243,311.851


In [11]:
filtered_dataset.columns.tolist()

['country',
 'year',
 'iso_code',
 'population',
 'gdp',
 'cement_co2',
 'cement_co2_per_capita',
 'co2',
 'co2_growth_abs',
 'co2_growth_prct',
 'co2_including_luc',
 'co2_including_luc_growth_abs',
 'co2_including_luc_growth_prct',
 'co2_including_luc_per_capita',
 'co2_including_luc_per_gdp',
 'co2_including_luc_per_unit_energy',
 'co2_per_capita',
 'co2_per_gdp',
 'co2_per_unit_energy',
 'coal_co2',
 'coal_co2_per_capita',
 'consumption_co2',
 'consumption_co2_per_capita',
 'consumption_co2_per_gdp',
 'cumulative_cement_co2',
 'cumulative_co2',
 'cumulative_co2_including_luc',
 'cumulative_coal_co2',
 'cumulative_flaring_co2',
 'cumulative_gas_co2',
 'cumulative_luc_co2',
 'cumulative_oil_co2',
 'cumulative_other_co2',
 'energy_per_capita',
 'energy_per_gdp',
 'flaring_co2',
 'flaring_co2_per_capita',
 'gas_co2',
 'gas_co2_per_capita',
 'ghg_excluding_lucf_per_capita',
 'ghg_per_capita',
 'land_use_change_co2',
 'land_use_change_co2_per_capita',
 'methane',
 'methane_per_capita',
 

In [12]:
filtered_dataset["calculated_co2_per_capita"] = (
    (filtered_dataset["co2"]*1000000) /
    filtered_dataset["population"]
)

##### Verifying the CO₂ Per Capita Feature

The existing `co2_per_capita` feature is verified by recalculating it using the `co2` and `population` columns. The values are compared after rounding to account for minor floating-point differences, and a sample of three countries across three years is displayed for validation.

In [13]:


comparison = pd.Series(np.isclose(
    filtered_dataset["co2_per_capita"],
    filtered_dataset["calculated_co2_per_capita"],
    atol=0.001)
)

comparison.value_counts()

True    350
Name: count, dtype: int64

In [14]:
mismatch = filtered_dataset[
    comparison == False
]

mismatch[
    [
        "country",
        "year",
        "co2",
        "population",
        "co2_per_capita",
        "calculated_co2_per_capita"
    ]
].head(20)

,country,year,co2,population,co2_per_capita,calculated_co2_per_capita


The recalculated values closely match the provided feature. Minor discrepancies are attributed to rounding and precision differences in the source dataset

In [15]:
verification_sample = filtered_dataset[
    (
        filtered_dataset["country"].isin(
            ["India", "China", "United States"]
        )
    )
    &
    (
        filtered_dataset["year"].isin(
            [1995, 2005, 2015]
        )
    )
]

verification_sample[
    [
        "country",
        "year",
        "co2",
        "population",
        "co2_per_capita",
        "calculated_co2_per_capita"
    ]
]

,country,year,co2,population,co2_per_capita,calculated_co2_per_capita
75,China,1995,3351.197,1.220134e+09,2.747,2.746581
85,China,2005,5881.991,1.310027e+09,4.490,4.489976
95,China,2015,9858.040,1.396134e+09,7.061,7.060955
145,India,1995,760.480,9.603010e+08,0.792,0.791918
155,India,2005,1195.393,1.154676e+09,1.035,1.035262
165,India,2015,2231.817,1.328024e+09,1.681,1.680554
320,United States,1995,5425.837,2.682058e+08,20.230,20.230126
330,United States,2005,6126.903,2.957167e+08,20.719,20.718829
340,United States,2015,5368.497,3.261265e+08,16.461,16.461395


##### Observation

The `co2_per_capita` feature was verified by recalculating CO₂ emissions per person using the `co2` and `population` columns for three countries across three different years. The recalculated values closely match the existing dataset values, with only very small differences caused by floating-point precision and rounding of the displayed data. This confirms that the `co2_per_capita` feature is correctly computed and can be used for further analysis.

In [16]:
filtered_dataset["ghg_intensity"] = (
    filtered_dataset["total_ghg"] /
    filtered_dataset["gdp"]
)

In [17]:
filtered_dataset[
    [
        "country",
        "year",
        "total_ghg",
        "gdp",
        "ghg_intensity"
    ]
].head(10)

,country,year,total_ghg,gdp,ghg_intensity
0,Australia,1990,538.479,4.641366e+11,1.160174e-09
1,Australia,1991,522.525,4.613021e+11,1.132718e-09
2,Australia,1992,568.962,4.786337e+11,1.188721e-09
3,Australia,1993,580.666,5.015248e+11,1.157801e-09
4,Australia,1994,554.084,5.279932e+11,1.049415e-09
5,Australia,1995,672.727,5.490387e+11,1.225281e-09
6,Australia,1996,677.191,5.747656e+11,1.178204e-09
7,Australia,1997,649.422,6.014728e+11,1.079720e-09
8,Australia,1998,705.618,6.348471e+11,1.111477e-09
9,Australia,1999,710.665,6.645207e+11,1.069440e-09


##### Creating the GHG Intensity Feature

A new feature called `ghg_intensity` is created to measure the amount of greenhouse gas emissions generated for each unit of economic output.

The feature is calculated by dividing `total_ghg` (measured in million tonnes) by `gdp` (measured in US dollars). Lower GHG intensity indicates a more carbon-efficient economy because it produces greater economic output for the same level of greenhouse gas emissions.

The feature is calculated only where both `total_ghg` and `gdp` values are available. Rows with missing values remain as `NaN`.

In [18]:
missing_gdp = filtered_dataset[
    filtered_dataset["gdp"].isna()
]

missing_gdp[
    [
        "country",
        "year"
    ]
].sort_values(["country","year"])

,country,year
33,Australia,2023
34,Australia,2024
68,Brazil,2023
69,Brazil,2024
103,China,2023
104,China,2024
138,Germany,2023
139,Germany,2024
173,India,2023
174,India,2024


##### Identifying Missing GDP Values

GHG intensity cannot be calculated when GDP data is unavailable. To identify these cases, rows with missing GDP values are extracted and displayed. This helps assess data completeness and explains why some `ghg_intensity` values remain missing.

In [19]:
filtered_dataset["co2_yoy_change"] = (
    filtered_dataset
    .groupby("country")["co2"]
    .diff()
)

##### Creating the Year-over-Year CO₂ Change Feature

A new feature called `co2_yoy_change` is created to measure the absolute change in CO₂ emissions from one year to the next for each country.

The feature is calculated by subtracting the previous year's CO₂ emissions from the current year's emissions using the `diff()` function. The dataset is grouped by country to ensure that changes are calculated independently for each country and not across different countries.

The first year for each country contains a `NaN` value because there is no previous year's data available for comparison.

In [20]:
filtered_dataset[
    [
        "country",
        "year",
        "co2",
        "co2_yoy_change"
    ]
].head(15)

,country,year,co2,co2_yoy_change
0,Australia,1990,278.061,NaN
1,Australia,1991,279.437,1.376
2,Australia,1992,284.433,4.996
3,Australia,1993,288.780,4.347
4,Australia,1994,293.613,4.833
5,Australia,1995,304.962,11.349
6,Australia,1996,311.851,6.889
7,Australia,1997,320.243,8.392
8,Australia,1998,334.038,13.795
9,Australia,1999,343.876,9.838


In [21]:
filtered_dataset["co2_yoy_pct_change"] = (
    filtered_dataset
    .groupby("country")["co2"]
    .pct_change() * 100
)

##### Creating the Year-over-Year Percentage Change Feature

A new feature called `co2_yoy_pct_change` is created to measure the percentage change in CO₂ emissions compared to the previous year for each country.

The feature is calculated using the `pct_change()` function, which computes the relative change between consecutive years. The dataset is grouped by country to ensure that percentage changes are calculated independently for each country.

The first observation for each country contains a `NaN` value because there is no previous year's data available for comparison.

In [22]:
filtered_dataset[
    [
        "country",
        "year",
        "co2",
        "co2_yoy_change",
        "co2_yoy_pct_change"
    ]
].head(15)

,country,year,co2,co2_yoy_change,co2_yoy_pct_change
0,Australia,1990,278.061,NaN,NaN
1,Australia,1991,279.437,1.376,0.494855
2,Australia,1992,284.433,4.996,1.787881
3,Australia,1993,288.780,4.347,1.528304
4,Australia,1994,293.613,4.833,1.673592
5,Australia,1995,304.962,11.349,3.865292
6,Australia,1996,311.851,6.889,2.258970
7,Australia,1997,320.243,8.392,2.691029
8,Australia,1998,334.038,13.795,4.307666
9,Australia,1999,343.876,9.838,2.945174


In [23]:


filtered_dataset["co2_yoy_pct_change"] = (
    filtered_dataset["co2_yoy_pct_change"]
    .replace([np.inf, -np.inf], np.nan)
)

pct_change() divides the current value difference by the previous value. If the previous value is zero, the calculation involves division by zero, which results in positive or negative infinity (inf or -inf). These values are typically replaced with NaN before further analysis.

In [24]:
top_growth_countries = (
    filtered_dataset
    .groupby("country")["co2_yoy_pct_change"]
    .mean()
    .sort_values(
        ascending=False
    )
    .head(5)
)

top_growth_countries

country
India           5.207068
China           4.905114
Brazil          2.463847
South Africa    1.132051
Australia       0.990026
Name: co2_yoy_pct_change, dtype: float64

###### Observation

The table shows the five countries with the highest average annual percentage growth in CO₂ emissions since 1990. Most of these are small countries or territories where even small increases in emissions result in large percentage changes. Therefore, percentage growth should be interpreted alongside absolute emission values for a complete understanding of emission trends.

In [25]:
top_reduction_countries = (
    filtered_dataset
    .groupby("country")["co2_yoy_change"]
    .mean()
    .sort_values(
        ascending=True
    )
    .head(5)
)

top_reduction_countries

country
Russia           -22.228235
Germany          -14.190500
United Kingdom    -8.501147
United States     -6.695324
Japan             -5.676059
Name: co2_yoy_change, dtype: float64

##### Identifying the Top 5 Countries with the Largest Average Annual CO₂ Reductions

The annual absolute change in CO₂ emissions is averaged for each country to identify long-term reduction trends. Countries with the most negative average year-over-year CO₂ change represent those that have consistently reduced emissions since 1990.

The countries are sorted in ascending order because larger reductions correspond to more negative values.

In [26]:
feature_columns = [
    "country",
    "year",
    "co2",
    "co2_per_capita",
    "co2_5yr_rolling_mean",
    "co2_lag1",
    "co2_lag2",
    "co2_lag3",
    "co2_yoy_pct_change",
    "ghg_intensity"
]

ghg_features = filtered_dataset[feature_columns]

In [27]:
ghg_features.head()

,country,year,co2,co2_per_capita,co2_5yr_rolling_mean,co2_lag1,co2_lag2,co2_lag3,co2_yoy_pct_change,ghg_intensity
0,Australia,1990,278.061,16.236,NaN,NaN,NaN,NaN,NaN,1.160174e-09
1,Australia,1991,279.437,16.103,NaN,278.061,NaN,NaN,0.494855,1.132718e-09
2,Australia,1992,284.433,16.208,NaN,279.437,278.061,NaN,1.787881,1.188721e-09
3,Australia,1993,288.780,16.294,NaN,284.433,279.437,278.061,1.528304,1.157801e-09
4,Australia,1994,293.613,16.405,284.8648,288.780,284.433,279.437,1.673592,1.049415e-09


In [28]:
ghg_features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   country               350 non-null    object 
 1   year                  350 non-null    int64  
 2   co2                   350 non-null    float64
 3   co2_per_capita        350 non-null    float64
 4   co2_5yr_rolling_mean  310 non-null    float64
 5   co2_lag1              340 non-null    float64
 6   co2_lag2              330 non-null    float64
 7   co2_lag3              320 non-null    float64
 8   co2_yoy_pct_change    340 non-null    float64
 9   ghg_intensity         330 non-null    float64
dtypes: float64(8), int64(1), object(1)
memory usage: 27.5+ KB


In [29]:
ghg_features.isna().sum()

country                  0
year                     0
co2                      0
co2_per_capita           0
co2_5yr_rolling_mean    40
co2_lag1                10
co2_lag2                20
co2_lag3                30
co2_yoy_pct_change      10
ghg_intensity           20
dtype: int64

In [30]:
ghg_features.to_csv(
    "data/ghg_features.csv",
    index=False
)

##### Creating the Final Feature Dataset

After completing the feature engineering process, a clean modelling dataset is created by selecting only the columns required for machine learning.

The dataset includes the target variable (`co2`) and the engineered features developed throughout Week 2. This final dataset will be used for training and evaluating machine learning models in the next stage of the project.

The `ghg_intensity` feature is included wherever GDP data is available. Missing values resulting from lag features, rolling averages, or unavailable GDP data are retained because they are expected outcomes of the feature engineering process.